# 04 - ML Model Pipeline

Complete ML workflow:
1. Prepare features and target
2. Train XGBoost and Random Forest
3. Evaluate and compare models
4. Register models
5. Use model predictions in a backtest

In [ ]:
import sys
sys.path.insert(0, '/app')

import pandas as pd
import matplotlib.pyplot as plt

from models.trainer import ModelTrainer
from models.xgboost_model import XGBoostModel
from models.random_forest_model import RandomForestModel
from models.registry import ModelRegistry

plt.style.use('seaborn-v0_8-darkgrid')

## 1. Prepare Training Data

In [ ]:
trainer = ModelTrainer()

feature_names = [
    'returns', 'log_returns', 'sma_20', 'ema_12', 'ema_26',
    'rsi_14', 'volatility_20', 'macd', 'macd_signal',
    'bb_upper', 'bb_lower'
]

X_train, X_test, y_train, y_test = trainer.prepare_data(
    ticker='AAPL',
    feature_names=feature_names,
    target_column='direction',  # 1 if next day up, 0 if down
    start_date='2020-01-01',
    test_size=0.2
)

print(f'Training set: {X_train.shape}')
print(f'Test set: {X_test.shape}')
print(f'Target distribution (train): {y_train.value_counts().to_dict()}')

## 2. Train XGBoost Model

In [ ]:
xgb_model = XGBoostModel(
    name='xgb_aapl_direction_v1',
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05
)

trainer.train_model(xgb_model, X_train, y_train)
xgb_metrics = trainer.evaluate_model(xgb_model, X_test, y_test)

print('XGBoost Metrics:')
for k, v in xgb_metrics.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')

## 3. Train Random Forest Model

In [ ]:
rf_model = RandomForestModel(
    name='rf_aapl_direction_v1',
    n_estimators=200,
    max_depth=10
)

trainer.train_model(rf_model, X_train, y_train)
rf_metrics = trainer.evaluate_model(rf_model, X_test, y_test)

print('Random Forest Metrics:')
for k, v in rf_metrics.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')

## 4. Compare Models

In [ ]:
registry = ModelRegistry()

# Register both models
training_info = {
    'ticker': 'AAPL',
    'features': feature_names,
    'start_date': '2020-01-01',
    'test_size': 0.2
}

registry.register(xgb_model, training_info, xgb_metrics)
registry.register(rf_model, training_info, rf_metrics)

# Compare
comparison = registry.compare(['xgb_aapl_direction_v1', 'rf_aapl_direction_v1'])
comparison

## 5. Feature Importance

In [ ]:
# XGBoost feature importance
import numpy as np

importance = xgb_model.model.feature_importances_
indices = np.argsort(importance)[::-1]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(len(importance)), importance[indices])
ax.set_yticks(range(len(importance)))
ax.set_yticklabels([feature_names[i] for i in indices])
ax.set_title('XGBoost Feature Importance')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 6. Cross-Validation

In [ ]:
X = pd.concat([X_train, X_test])
y = pd.concat([y_train, y_test])

cv_results = trainer.cross_validate(
    XGBoostModel(name='xgb_cv', n_estimators=200, max_depth=6, learning_rate=0.05),
    X, y, n_splits=5
)

print('Cross-Validation Results:')
for metric, values in cv_results.items():
    print(f'  {metric}: {np.mean(values):.4f} (+/- {np.std(values):.4f})')